In [ ]:
!pip install -q google-genai

In [ ]:
import sqlite3
import json

In [ ]:
from google import genai
from google.genai import types

In [ ]:
API_KEY = ""
client = genai.Client(api_key=API_KEY)

In [ ]:
conn = sqlite3.connect("placement.db")
cursor = conn.cursor()

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    cgpa REAL,
    skills TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS drives (
    id INTEGER PRIMARY KEY,
    company TEXT,
    role TEXT,
    min_cgpa REAL,
    department TEXT,
    status TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS interviews (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    company TEXT,
    slot TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS notifications (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    message TEXT
)
""")

cursor.execute("""
INSERT OR IGNORE INTO students
(id, name, department, cgpa, skills)
VALUES
(1, 'Rokesh', 'AI&DS', 8.2, 'Python, Java, SQL, Machine Learning')
""")

cursor.execute("""
INSERT OR IGNORE INTO drives
(id, company, role, min_cgpa, department, status)
VALUES
(1, 'Google', 'Software Engineer', 8.0, 'AI&DS', 'Open'),
(2, 'Microsoft', 'Data Analyst', 7.5, 'AI&DS', 'Open'),
(3, 'TCS', 'Graduate Engineer', 6.5, 'CSE', 'Open')
""")

conn.commit()

In [ ]:
def get_student(student_id):
    cursor.execute(
        "SELECT id, name, department, cgpa, skills FROM students WHERE id=?",
        (student_id,)
    )
    row = cursor.fetchone()

    if not row:
        return {"error": "Student not found"}

    return {
        "id": row[0],
        "name": row[1],
        "department": row[2],
        "cgpa": row[3],
        "skills": row[4]
    }


In [ ]:
def list_open_drives():
    cursor.execute("""
        SELECT id, company, role, min_cgpa, department
        FROM drives
        WHERE status='Open'
    """)

    rows = cursor.fetchall()

    return [
        {
            "id": r[0],
            "company": r[1],
            "role": r[2],
            "min_cgpa": r[3],
            "department": r[4]
        }
        for r in rows
    ]


In [ ]:
def check_eligibility(student_id, drive_id):
    student = get_student(student_id)

    if "error" in student:
        return student

    cursor.execute("""
        SELECT company, role, min_cgpa, department
        FROM drives
        WHERE id=?
    """, (drive_id,))

    drive = cursor.fetchone()

    if not drive:
        return {"error": "Drive not found"}

    company, role, min_cgpa, department = drive

    eligible = (
        student["cgpa"] >= min_cgpa and
        student["department"] == department
    )

    return {
        "student": student["name"],
        "company": company,
        "role": role,
        "eligible": eligible
    }

In [ ]:
def book_interview_slot(student_id, company, slot):
    cursor.execute("""
        INSERT INTO interviews(student_id, company, slot)
        VALUES (?, ?, ?)
    """, (student_id, company, slot))

    conn.commit()

    return {
        "success": True,
        "student_id": student_id,
        "company": company,
        "slot": slot
    }

In [ ]:
def notify_student(student_id, message):
    cursor.execute("""
        INSERT INTO notifications(student_id, message)
        VALUES (?, ?)
    """, (student_id, message))

    conn.commit()

    return {
        "success": True,
        "student_id": student_id,
        "message": message
    }

In [ ]:
tools = [
    {
        "name": "get_student",
        "description": "Get student placement profile.",
        "parameters": {
            "type": "object",
            "properties": {
                "student_id": {
                    "type": "integer"
                }
            },
            "required": ["student_id"]
        }
    },
    {
        "name": "list_open_drives",
        "description": "List all currently open placement drives.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "name": "check_eligibility",
        "description": "Check whether a student is eligible for a placement drive.",
        "parameters": {
            "type": "object",
            "properties": {
                "student_id": {
                    "type": "integer"
                },
                "drive_id": {
                    "type": "integer"
                }
            },
            "required": ["student_id", "drive_id"]
        }
    },
    {
        "name": "book_interview_slot",
        "description": "Book an interview slot for a student.",
        "parameters": {
            "type": "object",
            "properties": {
                "student_id": {
                    "type": "integer"
                },
                "company": {
                    "type": "string"
                },
                "slot": {
                    "type": "string"
                }
            },
            "required": ["student_id", "company", "slot"]
        }
    },
    {
        "name": "notify_student",
        "description": "Send a notification to a student.",
        "parameters": {
            "type": "object",
            "properties": {
                "student_id": {
                    "type": "integer"
                },
                "message": {
                    "type": "string"
                }
            },
            "required": ["student_id", "message"]
        }
    }
]

In [ ]:
def execute_tool(name, args):
    if name == "get_student":
        return get_student(args["student_id"])

    if name == "list_open_drives":
        return list_open_drives()

    if name == "check_eligibility":
        return check_eligibility(
            args["student_id"],
            args["drive_id"]
        )

    if name == "book_interview_slot":
        return book_interview_slot(
            args["student_id"],
            args["company"],
            args["slot"]
        )

    if name == "notify_student":
        return notify_student(
            args["student_id"],
            args["message"]
        )

    return {"error": "Unknown tool"}

In [ ]:
def placement_assistant(user_input):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction="""
You are a college placement assistant.
Student ID 1 is the current student.
Use the available tools when database information or actions are required.
Do not invent placement information.
Give simple and clear answers.
""",
            tools=tools
        )
    )

    while response.function_calls:
        parts = []

        for call in response.function_calls:
            result = execute_tool(
                call.name,
                dict(call.args)
            )

            parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response=result
                )
            )

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                user_input,
                response.candidates[0].content,
                types.Content(
                    role="user",
                    parts=parts
                )
            ],
            config=types.GenerateContentConfig(
                tools=tools
            )
        )

    return response.text

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Assistant: Goodbye!")
        break

    print("Assistant:", placement_assistant(user_input))

KeyboardInterrupt: Interrupted by user